# Demonstração do Grover em hardware IBM (E3, hipótese H5)

Executa a interface numa instância mínima (m = 3 arcos, custo unitário, k = 1, ótimo único) em três ambientes, nesta ordem: simulador ideal, simulador com ruído (`FakeFez`, modelo do processador Heron de 156 qubits) e, se liberado, hardware IBM real.

Duas variantes, com o **mesmo oráculo de tabela-verdade** (marca só a interdição ótima, limiar τ = F\* + 1):
- **interface:** preparação uniforme sobre F_1 = {x : |x| ≤ 1}, N = 4; a teoria dá P(ótimo) = sin²((2j+1)θ) com sin²θ = 1/4, ou seja, P = 1 com j = 1;
- **hipercubo:** H^⊗3, N = 8 (o orçamento só aparece no oráculo); P = 0,945 com j = 2.

**H5 (protocolo congelado):** no ambiente de hardware, a frequência da interdição ótima depois do número ideal de iterações é significativamente maior que 1/N (teste binomial unilateral, nível 0,05).

O oráculo de tabela-verdade custa exponencial em m: esta demonstração mostra a **amplificação**, não a vantagem.

## 1. Imports, instância e constantes

In [ ]:
import math

import numpy as np
import pandas as pd
from IPython.display import display
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeFez
from scipy.stats import binomtest

from qmfnip.grover import search_space
from qmfnip.instances import full_cut_cost, generate_random, max_flow
from qmfnip.oracle import (
    circuit_resources,
    free_plan_backend,
    grover_circuit,
    phase_oracle,
    sample_circuits,
    state_preparation,
)
from qmfnip.outputs import ExperimentOutput, capture_environment, latest_experiment
from qmfnip.plots import plot_amplification, plot_counts

In [ ]:
M_ARCS = 3
K = 1
SHOTS = 4096
ITERATIONS = (0, 1, 2, 3)
SEED = 2026
ALPHA = 0.05
ALLOW_HARDWARE = False
IBM_BACKEND_NAME = None
RECOVER_JOB_ID = None
REUSE_EXISTING = True
EXPERIMENT_TAG = "grover_ideal-grover_fake_fez-grover_ibm__random__n3-3__runs4096"
VARIANTS = {"interface": K, "hypercube": None}
TITLES = {"interface": "interface: Dicke preparation over $F_1$ (N = 4)", "hypercube": "hypercube: $H^{\\otimes 3}$ (N = 8)"}

Escolha automática da instância: a primeira semente do gerador aleatório que dá m = 3, custo unitário, k = 1 < κ, ótimo único (M = 1) e instância não degenerada (0 < F\* < fluxo original).

In [ ]:
for instance_seed in range(1000):
    instance = generate_random(3, instance_seed, arc_factor=10.0, cost_range=(1, 1), budget=K)
    if instance.m != M_ARCS or full_cut_cost(instance) <= K:
        continue
    space = search_space(instance)
    if space.optimal_count == 1 and 0 < space.optimum < max_flow(instance):
        break
optimum_bits = space.interdictions[space.optimal_indices[0]].astype(int).tolist()
OPTIMUM = "".join(map(str, optimum_bits))
FEASIBLE = {"".join(map(str, row)) for row in space.interdictions.astype(int).tolist()}
pd.DataFrame(
    {
        "arc": [f"{i}->{j}" for i, j in instance.arcs.tolist()],
        "capacity": instance.capacity,
        "in_optimum": optimum_bits,
    }
), {
    "instance_id": instance.instance_id,
    "seed": instance_seed,
    "flow_without_interdiction": max_flow(instance),
    "optimal_flow": space.optimum,
    "optimum": OPTIMUM,
    "kappa": full_cut_cost(instance),
    "feasible": sorted(FEASIBLE),
}

In [ ]:
oracle = phase_oracle(M_ARCS, [optimum_bits])
circuits, theory = [], []
for variant, k in VARIANTS.items():
    preparation = state_preparation(M_ARCS, k)
    size = len(FEASIBLE) if k is not None else 2**M_ARCS
    theta = math.asin(math.sqrt(1 / size))
    for iterations in ITERATIONS:
        circuit = grover_circuit(preparation, oracle, iterations)
        circuit.name = f"{variant}_j{iterations}"
        circuits.append(circuit)
        theory.append(
            {
                "variant": variant,
                "iterations": iterations,
                "N": size,
                "uniform": 1 / size,
                "probability": math.sin((2 * iterations + 1) * theta) ** 2,
            }
        )
theory = pd.DataFrame(theory)
BEST_ITERATIONS = theory.loc[theory.groupby("variant")["probability"].idxmax(), ["variant", "iterations"]].set_index("variant")["iterations"].to_dict()
display(theory)
BEST_ITERATIONS

## 2. Execução dos algoritmos e testes

Pasta do experimento: reaproveita a última com o mesmo nome se `REUSE_EXISTING` (para não refazer o que já rodou, sobretudo o job da IBM).

In [ ]:
try:
    output = latest_experiment(contains=EXPERIMENT_TAG) if REUSE_EXISTING else None
except FileNotFoundError:
    output = None
if output is None:
    output = ExperimentOutput.create(["grover_ideal", "grover_fake_fez", "grover_ibm"], "random", [instance.n], SHOTS)
    output.write_instance(instance)
    output.write_json("environment.json", capture_environment())
    output.update_manifest(
        {
            "status": "running",
            "instance": {instance.instance_id: instance.content_hash},
            "config": {
                "m": M_ARCS, "k": K, "shots": SHOTS, "iterations": list(ITERATIONS), "seed": SEED, "alpha": ALPHA,
                "optimum": OPTIMUM, "variants": {name: k for name, k in VARIANTS.items()},
            },
        }
    )
    output.write_table("theory", theory)
output.path.name

Recursos dos circuitos: lógicos (base `cx`, `u`) e transpilados para o `FakeFez`.

In [ ]:
resources = pd.DataFrame(
    [
        {"circuit": circuit.name, **{key: value for key, value in circuit_resources(circuit, backend).items() if key != "operations"}}
        for backend in (None, FakeFez())
        for circuit in circuits
    ]
)
output.write_table("circuit_resources", resources)
resources

### Simulador ideal

Aceite antes de seguir: cada frequência do ótimo a até 3 erros-padrão da teoria.

In [ ]:
def collect(result: dict, environment: str) -> pd.DataFrame:
    rows = []
    for circuit, counts in zip(circuits, result["counts"]):
        variant, iterations = circuit.name.rsplit("_j", 1)
        total = sum(counts.values())
        for state, value in counts.items():
            rows.append(
                {
                    "environment": environment, "variant": variant, "iterations": int(iterations), "state": state,
                    "count": value, "frequency": value / total, "shots": total,
                }
            )
    return pd.DataFrame(rows)


def run_or_load(environment: str, backend, allow_hardware: bool = False, job=None) -> pd.DataFrame:
    stored = output.path / f"counts_{environment}.csv"
    if stored.exists():
        return pd.read_csv(stored, dtype={"state": str})
    result = sample_circuits(
        circuits, backend, SHOTS, seed=SEED, allow_hardware=allow_hardware, job=job,
        on_submit=lambda submission: output.write_json(f"job_{environment}.json", submission),
    )
    frame = collect(result, environment)
    frame.to_csv(stored, index=False)
    output.write_json(f"run_{environment}.json", result["metadata"])
    return frame


ideal = run_or_load("ideal", AerSimulator())
check = (
    ideal[ideal["state"] == OPTIMUM][["variant", "iterations", "frequency"]]
    .merge(theory, on=["variant", "iterations"], how="right")
    .fillna({"frequency": 0.0})
)
check["tolerance"] = 3 * np.sqrt(check["probability"] * (1 - check["probability"]) / SHOTS) + 1 / SHOTS
check["ok"] = (check["frequency"] - check["probability"]).abs() <= check["tolerance"]
assert check["ok"].all(), check
check

### Simulador com ruído (`FakeFez`)

In [ ]:
fake = run_or_load("fake_fez", FakeFez())
fake[fake["state"] == OPTIMUM][["variant", "iterations", "frequency"]]

### Hardware IBM

Só roda com `ALLOW_HARDWARE = True` e a conta IBM configurada no terminal do usuário (o token nunca passa pelo notebook). Só usa instâncias do plano gratuito (Open Plan, 10 min/mês): `free_plan_backend` escolhe a instância gratuita e `sample_circuits` recusa enviar o job por qualquer instância paga. O id do job é gravado em `job_ibm.json` assim que o job é enviado. Se o kernel cair, ponha o id em `RECOVER_JOB_ID` e rode de novo esta célula.

In [ ]:
hardware = None
if ALLOW_HARDWARE or RECOVER_JOB_ID:
    from qiskit_ibm_runtime import QiskitRuntimeService

    if RECOVER_JOB_ID:
        job = QiskitRuntimeService().job(RECOVER_JOB_ID)
        backend = job.backend()
    else:
        job = None
        backend = free_plan_backend(IBM_BACKEND_NAME)
    hardware = run_or_load("ibm", backend, allow_hardware=True, job=job)
elif (output.path / "counts_ibm.csv").exists():
    hardware = pd.read_csv(output.path / "counts_ibm.csv", dtype={"state": str})
"no hardware run" if hardware is None else hardware[hardware["state"] == OPTIMUM][["variant", "iterations", "frequency"]]

## 3. Avaliação dos resultados, gráficos e tabelas

In [ ]:
environments = {"ideal simulator": ideal, "noisy simulator (FakeFez)": fake}
if hardware is not None:
    environments["IBM hardware"] = hardware
counts = pd.concat([frame.assign(environment=name) for name, frame in environments.items()], ignore_index=True)
grid = theory[["variant", "iterations", "N", "uniform", "probability"]].merge(
    pd.DataFrame({"environment": list(environments)}), how="cross"
)
optimum_hits = counts[counts["state"] == OPTIMUM][["environment", "variant", "iterations", "count", "frequency"]]
amplification = grid.merge(optimum_hits, on=["environment", "variant", "iterations"], how="left").fillna({"count": 0, "frequency": 0.0})
amplification["count"] = amplification["count"].astype(int)
amplification["feasible_frequency"] = [
    counts[(counts["environment"] == row.environment) & (counts["variant"] == row.variant) & (counts["iterations"] == row.iterations) & counts["state"].isin(FEASIBLE)]["frequency"].sum()
    for row in amplification.itertuples()
]
amplification["p_value"] = [
    binomtest(row.count, SHOTS, row.uniform, alternative="greater").pvalue for row in amplification.itertuples()
]
amplification["significant"] = amplification["p_value"] < ALPHA
output.write_table("amplification", amplification)
amplification

### Veredito da H5

Avaliado no número ideal de iterações de cada variante. O ambiente de decisão é o hardware IBM; sem hardware, o veredito fica **pendente**, e o resultado do `FakeFez` é reportado só como simulação com ruído.

In [ ]:
best = amplification[amplification.apply(lambda row: row["iterations"] == BEST_ITERATIONS[row["variant"]], axis=1)]
decision = "IBM hardware" if hardware is not None else None
h5 = {
    "decision_environment": decision,
    "rows": best[["environment", "variant", "iterations", "probability", "frequency", "p_value", "significant"]].to_dict(orient="records"),
}
if decision is None:
    h5["verdict"] = "pending (no hardware run)"
else:
    decisive = best[best["environment"] == decision]
    h5["verdict"] = "not refuted" if decisive["significant"].all() else "refuted"
output.write_json("h5.json", h5)
display(best)
h5["verdict"]

### Figura F4: probabilidade do ótimo × iterações

In [ ]:
figure = plot_amplification(amplification, theory, SHOTS, titles=TITLES, path=output.figure_path("f4_amplification"))

### Figura F5: distribuição das medições no número ideal de iterações

In [ ]:
panels = pd.concat(
    [
        counts[(counts["variant"] == variant) & (counts["iterations"] == iterations)].assign(panel=variant)
        for variant, iterations in BEST_ITERATIONS.items()
    ],
    ignore_index=True,
)
figure = plot_counts(panels, OPTIMUM, FEASIBLE, titles={key: f"{TITLES[key]}, j = {value}" for key, value in BEST_ITERATIONS.items()}, path=output.figure_path("f5_counts"))

### Tabela T4: execuções

In [ ]:
def job_times(metadata: dict) -> dict:
    metrics = metadata.get("metrics")
    if not isinstance(metrics, dict):
        return {"qpu_seconds": None, "queue_seconds": None, "total_seconds": None}
    stamps = {key: pd.Timestamp(value) for key, value in metrics["timestamps"].items() if value}
    return {
        "qpu_seconds": metrics.get("usage", {}).get("qpu_charge_time_seconds"),
        "queue_seconds": (stamps["running"] - stamps["created"]).total_seconds(),
        "total_seconds": (stamps["finished"] - stamps["created"]).total_seconds(),
    }


rows = []
for environment, key in (("ideal simulator", "ideal"), ("noisy simulator (FakeFez)", "fake_fez"), ("IBM hardware", "ibm")):
    metadata = output.read_json(f"run_{key}.json")
    if metadata is None:
        continue
    for item, circuit in zip(metadata["transpiled"], circuits):
        variant, iterations = circuit.name.rsplit("_j", 1)
        if int(iterations) != BEST_ITERATIONS[variant]:
            continue
        hit = amplification[(amplification["environment"] == environment) & (amplification["variant"] == variant) & (amplification["iterations"] == int(iterations))].iloc[0]
        rows.append(
            {
                "environment": environment, "backend": metadata["backend"], "job_id": metadata["job_id"],
                "date": metadata["finished_at"], "variant": variant, "iterations": int(iterations), "shots": SHOTS,
                "depth": item["depth"], "two_qubit_gates": item["two_qubit_gates"],
                **job_times(metadata), "theory": hit["probability"], "frequency": hit["frequency"],
                "uniform": hit["uniform"], "p_value": hit["p_value"],
            }
        )
t4 = pd.DataFrame(rows)
output.write_table("t4_hardware", t4)
output.update_manifest({"status": "completed" if hardware is not None else "simulators_only"})
t4